In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install h5py xarray rasterio numpy pandas affine

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 84.2 MB/s eta 0:00:00


In [ ]:
import h5py
import numpy as np
import rasterio
from affine import Affine

# Paths to your HDF5 files
h5_vel = '/content/drive/MyDrive/mintpy_output/location4/velocity.h5'
h5_ts  = '/content/drive/MyDrive/mintpy_output/location4/timeseries.h5'

# 1. EXPORT VELOCITY MAP
with h5py.File(h5_vel, 'r') as f:
    vel = f['velocity'][:]                # 2D array (rows, cols)
    # Read geospatial metadata (cast to float if stored as bytes)
    x0 = float(f.attrs['X_FIRST'])
    y0 = float(f.attrs['Y_FIRST'])
    dx = float(f.attrs['X_STEP'])
    dy = float(f.attrs['Y_STEP'])
    rows, cols = vel.shape

# Build the affine transform: Affine(scale_x, 0, x_origin, 0, scale_y, y_origin)
transform = Affine(dx, 0.0, x0,
                   0.0, dy, y0)

# Define common raster metadata
meta = {
    'driver': 'GTiff',
    'dtype': 'float32',
    'count': 1,
    'height': rows,
    'width': cols,
    'crs': 'EPSG:4326',
    'transform': transform
}

# Write velocity_aligned_L4.tif
with rasterio.open('velocity_aligned_L4.tif', 'w', **meta) as dst:
    dst.write(vel.astype('float32'), 1)

print("Velocity map written: velocity_aligned_L4.tif")

# 2. EXPORT TREND MAP
with h5py.File(h5_ts, 'r') as f:
    ts = f['timeseries'][:]              # 3D array (time, rows, cols)
    raw_dates = f['date'][:]             # byte‐strings like b'20230707'

# Decode dates and convert to numpy datetime64 (daily)
dates = np.array([np.datetime64(d.decode('utf-8'), 'D') for d in raw_dates])
n_times = len(dates)

# Prepare the time axis for least squares
t = np.arange(n_times, dtype=float)
mean_t = t.mean()
var_t = ((t - mean_t) ** 2).sum()

# Calculate per-pixel slope: sum((value * (t - mean_t)), axis=0) / var(t)
# Here ts has shape (time, rows, cols); broadcasting works naturally
trend = ((ts * (t[:, None, None] - mean_t)).sum(axis=0)) / var_t

# Write trend_L4.tif using the same metadata
with rasterio.open('trend_L4.tif', 'w', **meta) as dst:
    dst.write(trend.astype('float32'), 1)

print("Trend map written: trend_L4.tif")

Velocity map written: velocity_aligned_L4.tif
Trend map written: trend_L4.tif


In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

# Paths
dem_src = '/content/S1AA_20220101T004848_20220113T004847_VVP012_INT80_G_weF_5C84_dem.tif'
dem_dst = 'DEM_L4.tif'

# Reproject DEM to match velocity grid
with rasterio.open('velocity_aligned_L4.tif') as ref:
    dst_crs = ref.crs
    dst_transform = ref.transform
    dst_width, dst_height = ref.width, ref.height

with rasterio.open(dem_src) as src:
    transform2, width2, height2 = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds)
    meta2 = src.meta.copy()
    meta2.update({
        'crs': dst_crs,
        'transform': dst_transform,
        'width': dst_width,
        'height': dst_height
    })
    with rasterio.open(dem_dst, 'w', **meta2) as dst:
        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=dst_transform, dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )

# Compute slope and aspect
import numpy as np

with rasterio.open(dem_dst) as src:
    dem = src.read(1)
    transform = src.transform
    dx = transform.a
    dy = -transform.e

# Calculate gradients (z_x, z_y)
dzdx, dzdy = np.gradient(dem, dx, dy)
slope = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))
aspect = (np.degrees(np.arctan2(dzdy, -dzdx)) + 180) % 360

meta3 = src.meta.copy()
for arr, name in [(slope, 'slope_L4.tif'), (aspect, 'aspect_L4.tif')]:
    with rasterio.open(name, 'w', **meta3) as dst:
        dst.write(arr.astype('float32'), 1)

In [ ]:
import rasterio

# Update these paths to point to your actual files in Drive
velocity_path = '/content/velocity_aligned_L4.tif'
dem_path      = '/content/S1AA_20220101T004848_20220113T004847_VVP012_INT80_G_weF_5C84_dem.tif'

def print_geospatial_metadata(path):
    with rasterio.open(path) as src:
        print(f"Metadata for {path}:")
        print("  CRS:          ", src.crs)
        print("  Transform:    ", src.transform)
        print("  Pixel Size:   ", (src.transform.a, -src.transform.e))
        print("  Width × Height:", src.width, "×", src.height)
        print("  Bounds:       ", src.bounds)
        print()

# Print metadata for both rasters
print_geospatial_metadata(velocity_path)
print_geospatial_metadata(dem_path)

Metadata for /content/velocity_aligned_L4.tif:
  CRS:           EPSG:4326
  Transform:     | 80.00, 0.00, 492600.00|
| 0.00,-80.00, 1399000.00|
| 0.00, 0.00, 1.00|
  Pixel Size:    (80.0, 80.0)
  Width × Height: 2072 × 2765
  Bounds:        BoundingBox(left=492600.0, bottom=1177800.0, right=658360.0, top=1399000.0)

Metadata for /content/S1AA_20220101T004848_20220113T004847_VVP012_INT80_G_weF_5C84_dem.tif:
  CRS:           EPSG:32643
  Transform:     | 80.00, 0.00, 375720.00|
| 0.00,-80.00, 1401720.00|
| 0.00, 0.00, 1.00|
  Pixel Size:    (80.0, 80.0)
  Width × Height: 3549 × 2605
  Bounds:        BoundingBox(left=375720.0, bottom=1193320.0, right=659640.0, top=1401720.0)



In [ ]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# 1. Read template (velocity_aligned_L4.tif) metadata
template_path = '/content/velocity_aligned_L4.tif'
with rasterio.open(template_path) as tpl:
    tpl_crs      = tpl.crs
    tpl_transform= tpl.transform
    tpl_width    = tpl.width
    tpl_height   = tpl.height
    tpl_meta     = tpl.meta.copy()

# 2. Open raw DEM
dem_src_path = '/content/S1AA_20220101T004848_20220113T004847_VVP012_INT80_G_weF_5C84_dem.tif'
with rasterio.open(dem_src_path) as src:
    dem_data      = src.read(1)
    src_transform = src.transform
    src_crs       = src.crs

# 3. Prepare output array
aligned_dem = np.empty((tpl_height, tpl_width), dtype=dem_data.dtype)
aligned_dem.fill(src.nodata if src.nodata is not None else np.nan)

# 4. Reproject & warp DEM into the velocity grid
reproject(
    source      = dem_data,
    destination = aligned_dem,
    src_transform= src_transform,
    src_crs     = src_crs,
    dst_transform= tpl_transform,
    dst_crs     = tpl_crs,
    resampling  = Resampling.bilinear
)

# 5. Write the aligned DEM
out_meta = tpl_meta
out_meta.update({
    'dtype':      aligned_dem.dtype,
    'count':      1,
    'nodata':     src.nodata
})
out_path = 'DEM_aligned_L4.tif'
with rasterio.open(out_path, 'w', **out_meta) as dst:
    dst.write(aligned_dem, 1)

print(f"Written aligned DEM to {out_path}")

Written aligned DEM to DEM_aligned_L4.tif


In [ ]:
import numpy as np
import rasterio

# Paths
dem_path   = 'DEM_aligned_L4.tif'
slope_path = 'slope_aligned_L4.tif'
asp_path   = 'aspect_aligned_L4.tif'

# Read the aligned DEM
with rasterio.open(dem_path) as src:
    dem       = src.read(1)
    meta      = src.meta.copy()
    transform = src.transform

# Compute pixel resolution in X and Y
dx = transform.a           # pixel width
dy = -transform.e          # pixel height (negative in transform)

# Calculate gradients
dzdx, dzdy = np.gradient(dem, dx, dy)

# Compute slope (degrees)
slope = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))

# Compute aspect (degrees clockwise from north)
aspect = (np.degrees(np.arctan2(dzdy, -dzdx)) + 180) % 360

# Update metadata for output (single band, float32)
meta.update(dtype='float32', count=1, nodata=None)

# Write slope
with rasterio.open(slope_path, 'w', **meta) as dst:
    dst.write(slope.astype('float32'), 1)

# Write aspect
with rasterio.open(asp_path, 'w', **meta) as dst:
    dst.write(aspect.astype('float32'), 1)

print("Slope written to:", slope_path)
print("Aspect written to:", asp_path)

Slope written to: slope_aligned_L4.tif
Aspect written to: aspect_aligned_L4.tif


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# Paths to your NetCDF rainfall files
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]

# Load + concatenate along time
rain = xr.open_mfdataset(era5_files, concat_dim='time', combine='nested', decode_cf=True)

# Swap valid_time → time if needed
if 'valid_time' in rain.dims:
    rain = (
        rain.swap_dims({'valid_time': 'time'})
            .assign_coords(time=rain.coords['valid_time'])
            .drop_vars('valid_time')
    )

# Sort and convert from m to mm
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

# Optionally clip to your bbox
# rain = rain.sel(latitude=slice(32.92,30.87), longitude=slice(75.66,78.74))

# Load InSAR dates
with rasterio.open('velocity_aligned_L4.tif') as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_width, tpl_height = tpl.width, tpl.height

insar_dates = ['20230707','20230719','20230731', …]  # your 38 dates
insar_dt    = [pd.to_datetime(d, format='%Y%m%d') for d in insar_dates]

# Loop to compute and export
for dt in insar_dt:
    start = dt - pd.Timedelta(days=11)
    window = rain.sel(time=slice(start, dt))
    if window.time.size < 12:
        print(f"Skipping {dt.date()} — only {window.time.size} days")
        continue

    # Sum over time
    rain12 = window['tp'].sum(dim='time').values.astype('float32')

    # Prepare output array on master grid
    out_arr = np.full((tpl_height, tpl_width), np.nan, dtype='float32')

    # Build source transform from the rain DataArray
    lat = window.latitude.values
    lon = window.longitude.values
    src_transform = rasterio.transform.from_bounds(
        lon.min(), lat.min(), lon.max(), lat.max(),
        len(lon), len(lat)
    )

    # Reproject into the InSAR grid
    reproject(
        source      = rain12,
        destination = out_arr,
        src_transform = src_transform,
        src_crs     = 'EPSG:4326',
        dst_transform= tpl_transform,
        dst_crs     = tpl_crs,
        resampling  = Resampling.bilinear
    )

    # Write GeoTIFF
    out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
    with rasterio.open(out_name, 'w', **tpl_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Wrote {out_name}")


SyntaxError: invalid character '…' (U+2026) (<ipython-input-6-432e004f36d4>, line 38)

In [ ]:
import h5py
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# 1. Load and merge ERA5 daily-sum rainfall NetCDFs
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]
rain = xr.open_mfdataset(era5_files, concat_dim='time', combine='nested', decode_cf=True)

# 2. Fix the time dimension if it’s named 'valid_time'
if 'valid_time' in rain.dims:
    rain = (
        rain.swap_dims({'valid_time': 'time'})
            .assign_coords(time=rain.coords['valid_time'])
            .drop_vars('valid_time')
    )

# 3. Ensure chronological order and convert from meters to millimeters
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

# 4. Read InSAR dates from your timeseries.h5 (38 dates)
with h5py.File('/content/drive/MyDrive/mintpy_output/location4/timeseries.h5', 'r') as f:
    insar_dates = [d.decode('utf-8') for d in f['date'][:]]
insar_dt = [pd.to_datetime(d, format='%Y%m%d') for d in insar_dates]

# 5. Prepare template grid from your aligned velocity GeoTIFF
with rasterio.open('/content/velocity_aligned_L4.tif') as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_w, tpl_h  = tpl.width, tpl.height

# 6. Loop over each InSAR date, sum the preceding 12 days, reproject, and export
for dt in insar_dt:
    start = dt - pd.Timedelta(days=11)
    window = rain.sel(time=slice(start, dt))
    if window.time.size < 12:
        print(f"Skipping {dt.date()}: only {window.time.size} days available")
        continue

    # Sum daily totals over the 12-day window
    rain12 = window['tp'].sum(dim='time').values.astype('float32')

    # Create an output array on the template grid
    out_arr = np.full((tpl_h, tpl_w), np.nan, dtype='float32')

    # Build source transform from the rainfall DataArray
    lat = window.latitude.values
    lon = window.longitude.values
    src_transform = rasterio.transform.from_bounds(
        lon.min(), lat.min(), lon.max(), lat.max(),
        len(lon), len(lat)
    )

    # Reproject rainfall sum into your InSAR grid
    reproject(
        source       = rain12,
        destination  = out_arr,
        src_transform= src_transform,
        src_crs      = 'EPSG:4326',
        dst_transform= tpl_transform,
        dst_crs      = tpl_crs,
        resampling   = Resampling.bilinear
    )

    # Write out the 12-day GeoTIFF
    out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
    with rasterio.open(out_name, 'w', **tpl_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Wrote {out_name}")

print(f"\nCompleted exports for {len(insar_dt)} InSAR dates.")

/usr/local/lib/python3.11/dist-packages/xarray/namedarray/core.py:503: UserWarning: Duplicate dimension names present: dimensions {'time'} appear more than once in dims=('time', 'time', 'latitude', 'longitude'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  self._dims = self._parse_dimensions(value)


ValueError: conflicting sizes for dimension 'time': length 550 on 'tp' and length 2 on {'time': 'tp'}

In [ ]:
import os
import h5py
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# 1. Paths to your daily‐sum ERA5 NetCDFs
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]

# 2. Open and merge on the 'time' coordinate
rain = xr.open_mfdataset(
    era5_files,
    concat_dim='time',
    combine='by_coords',
    decode_cf=True
)

# 3. If there's a 'valid_time' dimension, swap it to 'time'
if 'valid_time' in rain.dims:
    # Drop any conflicting 'time' coordinate first
    if 'time' in rain.coords and rain.coords['time'].shape != (rain.dims['valid_time'],):
        rain = rain.drop_vars('time')
    # Now swap and set the new time coord
    rain = (
        rain
        .swap_dims({'valid_time': 'time'})
        .assign_coords(time=rain.coords['valid_time'])
        .drop_vars('valid_time')
    )

# 4. Chronological order and unit conversion (m → mm)
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

# 5. Load InSAR dates from your timeseries.h5
with h5py.File('/content/drive/MyDrive/minty_output/location4/timeseries.h5', 'r') as f:
    insar_dates = [d.decode('utf-8') for d in f['date'][:]]
insar_dt = pd.to_datetime(insar_dates, format='%Y%m%d')

# 6. Read your velocity GeoTIFF as the template grid
template_path = '/content/velocity_aligned_L4.tif'
with rasterio.open(template_path) as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_w, tpl_h  = tpl.width, tpl.height

# 7. Compute and export each 12-day sum
output_dir = 'rainfall_12day'
os.makedirs(output_dir, exist_ok=True)

for dt in insar_dt:
    start = dt - pd.Timedelta(days=11)
    window = rain.sel(time=slice(start, dt))
    if window.time.size < 12:
        print(f"Skipping {dt.date()}: only {window.time.size} days available")
        continue

    # Sum over the 12-day window
    arr12 = window['tp'].sum(dim='time').values.astype('float32')

    # Prepare an empty array on the template grid
    out_arr = np.full((tpl_h, tpl_w), np.nan, dtype='float32')

    # Build source transform from rain coords
    lat = window.latitude.values
    lon = window.longitude.values
    src_transform = from_bounds(
        lon.min(), lat.min(), lon.max(), lat.max(),
        len(lon), len(lat)
    )

    # Reproject into the InSAR grid
    reproject(
        source       = arr12,
        destination  = out_arr,
        src_transform= src_transform,
        src_crs      = 'EPSG:4326',
        dst_transform= tpl_transform,
        dst_crs      = tpl_crs,
        resampling   = Resampling.bilinear
    )

    # Write the GeoTIFF
    out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
    out_path = os.path.join(output_dir, out_name)
    with rasterio.open(out_path, 'w', **tpl_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Wrote {out_name}")

print(f"\nCompleted exports for {len(insar_dt)} dates.")

ValueError: When combine='by_coords', passing a value for `concat_dim` has no effect. To manually combine along a specific dimension you should instead specify combine='nested' along with a value for `concat_dim`.

In [ ]:
import os
import h5py
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# 1. Paths to your daily‐sum ERA5 NetCDFs
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]

# 2. Open and merge along the 'time' dimension
rain = xr.open_mfdataset(
    era5_files,
    concat_dim='time',
    combine='nested',
    decode_cf=True
)

# 3. Swap valid_time → time if present
if 'valid_time' in rain.dims:
    # Drop any conflicting 'time' coord
    if 'time' in rain.coords and rain.coords['time'].shape != (rain.dims['valid_time'],):
        rain = rain.drop_vars('time')
    rain = (
        rain
        .swap_dims({'valid_time': 'time'})
        .assign_coords(time=rain.coords['valid_time'])
        .drop_vars('valid_time')
    )

# 4. Sort chronologically and convert from m to mm
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

# 5. Read InSAR dates from timeseries.h5
with h5py.File('/content/drive/MyDrive/minty_output/location4/timeseries.h5', 'r') as f:
    insar_dates = [d.decode('utf-8') for d in f['date'][:]]
insar_dt = pd.to_datetime(insar_dates, format='%Y%m%d')

# 6. Open velocity_aligned_L4.tif as your template grid
template_path = '/content/velocity_aligned_L4.tif'
with rasterio.open(template_path) as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_w, tpl_h  = tpl.width, tpl.height

# 7. Compute and export each 12-day sum
output_dir = '/content/drive/MyDrive/Updated_location4/rainfall_12day'
os.makedirs(output_dir, exist_ok=True)

for dt in insar_dt:
    start = dt - pd.Timedelta(days=11)
    window = rain.sel(time=slice(start, dt))
    if window.time.size < 12:
        print(f"Skipping {dt.date()}: only {window.time.size} days available")
        continue

    # Sum over the 12-day window
    arr12 = window['tp'].sum(dim='time').values.astype('float32')

    # Prepare an empty array on the template grid
    out_arr = np.full((tpl_h, tpl_w), np.nan, dtype='float32')

    # Build source transform from rainfall coords
    lat = window.latitude.values
    lon = window.longitude.values
    src_transform = from_bounds(
        lon.min(), lat.min(), lon.max(), lat.max(),
        len(lon), len(lat)
    )

    # Reproject into the InSAR grid
    reproject(
        source       = arr12,
        destination  = out_arr,
        src_transform= src_transform,
        src_crs      = 'EPSG:4326',
        dst_transform= tpl_transform,
        dst_crs      = tpl_crs,
        resampling   = Resampling.bilinear
    )

    # Write the GeoTIFF
    out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
    with rasterio.open(os.path.join(output_dir, out_name), 'w', **tpl_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Wrote {out_name}")

print(f"\nCompleted exports for {len(insar_dt)} dates.")

/usr/local/lib/python3.11/dist-packages/xarray/namedarray/core.py:503: UserWarning: Duplicate dimension names present: dimensions {'time'} appear more than once in dims=('time', 'time', 'latitude', 'longitude'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  self._dims = self._parse_dimensions(value)


ValueError: conflicting sizes for dimension 'time': length 550 on 'tp' and length 2 on {'time': 'tp'}

In [ ]:
import os
import h5py
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# 1. Paths to your ERA5 NetCDFs (daily total precipitation)
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]

# 2. Open and concatenate along time
rain = xr.open_mfdataset(
    era5_files,
    concat_dim='time',
    combine='nested',
    decode_cf=True
)

# 3. If ERA5 used a 'valid_time' dimension, rename it to 'time' cleanly
if 'valid_time' in rain.dims:
    # 3a. Pull out the old coordinate values
    old_time_coord = rain.coords['valid_time']
    # 3b. Rename the dimension itself
    rain = rain.rename_dims({'valid_time': 'time'})
    # 3c. Drop the leftover 'valid_time' coordinate variable
    rain = rain.drop_vars('valid_time')
    # 3d. Re-assign a proper time coordinate
    rain = rain.assign_coords(time=old_time_coord)

# 4. Sort chronologically and convert precipitation from m → mm
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

# 5. Load your 38 InSAR dates from timeseries.h5
with h5py.File('/content/drive/MyDrive/minty_output/location4/timeseries.h5', 'r') as f:
    insar_dates = [d.decode('utf-8') for d in f['date'][:]]
insar_dt = pd.to_datetime(insar_dates, format='%Y%m%d')

# 6. Read your velocity GeoTIFF as the template grid
template_path = '/content/velocity_aligned_L4.tif'
with rasterio.open(template_path) as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_w, tpl_h  = tpl.width, tpl.height

# 7. Create output directory
output_dir = '/content/drive/MyDrive/Updated_location4/rainfall_12day'
os.makedirs(output_dir, exist_ok=True)

# 8. Loop over each InSAR date, sum the prior 12 days, reproject, and export
for dt in insar_dt:
    start = dt - pd.Timedelta(days=11)
    window = rain.sel(time=slice(start, dt))
    if window.time.size < 12:
        print(f"Skipping {dt.date()}: only {window.time.size} days available")
        continue

    # Sum over the 12-day window
    arr12 = window['tp'].sum(dim='time').values.astype('float32')

    # Prepare an empty array on the template grid
    out_arr = np.full((tpl_h, tpl_w), np.nan, dtype='float32')

    # Build source transform from the rainfall lat/lon grid
    lat = window.latitude.values
    lon = window.longitude.values
    src_transform = from_bounds(
        lon.min(), lat.min(),
        lon.max(), lat.max(),
        len(lon), len(lat)
    )

    # Reproject into the InSAR grid
    reproject(
        source        = arr12,
        destination   = out_arr,
        src_transform = src_transform,
        src_crs       = 'EPSG:4326',
        dst_transform = tpl_transform,
        dst_crs       = tpl_crs,
        resampling    = Resampling.bilinear
    )

    # Write the GeoTIFF
    out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
    out_path = os.path.join(output_dir, out_name)
    with rasterio.open(out_path, 'w', **tpl_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Wrote {out_name}")

print(f"\nCompleted exports for {len(insar_dt)} dates.")

ValueError: Cannot rename valid_time to time because time already exists. Try using swap_dims instead.

In [ ]:
import os
import h5py
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# 1. Load ERA5 daily‐sum NetCDFs
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]
rain = xr.open_mfdataset(
    era5_files,
    concat_dim='time',
    combine='nested',
    decode_cf=True
)

# 2. If there's a valid_time dim, swap it to time without conflict
if 'valid_time' in rain.dims:
    # a) drop any existing time coordinate that conflicts
    if 'time' in rain.coords and rain.coords['time'].shape != (rain.dims['valid_time'],):
        rain = rain.drop_vars('time')
    # b) swap and reassign as a datetime64 index
    rain = rain.swap_dims({'valid_time': 'time'})
    rain = rain.assign_coords(time=pd.to_datetime(rain.indexes['time'].values))
    rain = rain.drop_vars('valid_time')

# 3. Sort by time and convert precipitation to millimeters
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

# 4. Read InSAR dates
with h5py.File('/content/drive/MyDrive/minty_output/location4/timeseries.h5', 'r') as f:
    insar_dates = [d.decode() for d in f['date'][:]]
insar_dt = pd.to_datetime(insar_dates, format='%Y%m%d')

# 5. Load velocity_aligned_L4.tif as the template grid
tpl_path = '/content/velocity_aligned_L4.tif'
with rasterio.open(tpl_path) as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_w, tpl_h  = tpl.width, tpl.height

# 6. Compute and export 12-day sums for each InSAR date
out_dir = '/content/drive/MyDrive/Updated_location4/rainfall_12day'
os.makedirs(out_dir, exist_ok=True)

for dt in insar_dt:
    window = rain.sel(time=slice(dt - pd.Timedelta(days=11), dt))
    if window.time.size < 12:
        print(f"Skipping {dt.date()}: only {window.time.size} days")
        continue

    # sum the 12-day daily totals
    arr12 = window['tp'].sum(dim='time').values.astype('float32')

    # prepare an empty array on the template grid
    out_arr = np.full((tpl_h, tpl_w), np.nan, dtype='float32')

    # build the source transform from rainfall coords
    lat = window.latitude.values
    lon = window.longitude.values
    src_transform = from_bounds(
        lon.min(), lat.min(), lon.max(), lat.max(),
        len(lon), len(lat)
    )

    # reproject rainfall sum into InSAR grid
    reproject(
        source        = arr12,
        destination   = out_arr,
        src_transform = src_transform,
        src_crs       = 'EPSG:4326',
        dst_transform = tpl_transform,
        dst_crs       = tpl_crs,
        resampling    = Resampling.bilinear
    )

    # write the GeoTIFF
    out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
    with rasterio.open(os.path.join(out_dir, out_name), 'w', **tpl_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Wrote {out_name}")

print(f"Completed exports for {len(insar_dt)} dates.")

/usr/local/lib/python3.11/dist-packages/xarray/namedarray/core.py:503: UserWarning: Duplicate dimension names present: dimensions {'time'} appear more than once in dims=('time', 'time', 'latitude', 'longitude'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  self._dims = self._parse_dimensions(value)


ValueError: conflicting sizes for dimension 'time': length 550 on 'tp' and length 2 on {'time': 'tp'}

In [ ]:
import os
import h5py
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# 1. Paths to your ERA5 daily-sum NetCDFs
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]

# 2. Open and concatenate along 'time'
rain = xr.open_mfdataset(
    era5_files,
    concat_dim='time',
    combine='nested',
    decode_cf=True
)

# 3. If there's a 'valid_time' dimension, avoid duplicate 'time' dims:
if 'valid_time' in rain.dims:
    # a) Rename the existing 'time' dim so it won't collide
    rain = rain.rename_dims({'time': 'old_time'})
    # b) Drop its coordinate variable
    if 'old_time' in rain.coords:
        rain = rain.drop_vars('old_time')
    # c) Now swap 'valid_time' → 'time' and drop the old 'valid_time'
    rain = (
        rain
        .swap_dims({'valid_time': 'time'})
        .assign_coords(time=rain.coords['time'])
        .drop_vars('valid_time')
    )

# 4. Sort chronologically and convert from meters to millimeters
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

# 5. Read your 38 InSAR dates from timeseries.h5
with h5py.File('/content/drive/MyDrive/minty_output/location4/timeseries.h5', 'r') as f:
    insar_dates = [d.decode('utf-8') for d in f['date'][:]]
insar_dt = pd.to_datetime(insar_dates, format='%Y%m%d')

# 6. Load your velocity GeoTIFF as the template grid
template_path = '/content/velocity_aligned_L4.tif'
with rasterio.open(template_path) as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_w, tpl_h  = tpl.width, tpl.height

# 7. Compute and export each 12-day sum
output_dir = '/content/drive/MyDrive/Updated_location4/rainfall_12day'
os.makedirs(output_dir, exist_ok=True)

for dt in insar_dt:
    start = dt - pd.Timedelta(days=11)
    window = rain.sel(time=slice(start, dt))
    if window.time.size < 12:
        print(f"Skipping {dt.date()}: only {window.time.size} days available")
        continue

    # Sum daily totals over the 12-day window
    arr12 = window['tp'].sum(dim='time').values.astype('float32')

    # Prepare an empty array on the template grid
    out_arr = np.full((tpl_h, tpl_w), np.nan, dtype='float32')

    # Build source transform from the rainfall coords
    lat = window.latitude.values
    lon = window.longitude.values
    src_transform = from_bounds(
        lon.min(), lat.min(), lon.max(), lat.max(),
        len(lon), len(lat)
    )

    # Reproject into the InSAR grid
    reproject(
        source        = arr12,
        destination   = out_arr,
        src_transform = src_transform,
        src_crs       = 'EPSG:4326',
        dst_transform = tpl_transform,
        dst_crs       = tpl_crs,
        resampling    = Resampling.bilinear
    )

    # Write the GeoTIFF
    out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
    out_path = os.path.join(output_dir, out_name)
    with rasterio.open(out_path, 'w', **tpl_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Wrote {out_name}")

print(f"\nCompleted exports for {len(insar_dt)} dates.")

KeyError: "No variable named 'time'. Variables on the dataset include ['tp', 'latitude', 'longitude', 'valid_time', 'number']"

In [ ]:
import os
import h5py
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# 1. Load your two daily‐sum ERA5 NetCDFs
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]
rain = xr.open_mfdataset(era5_files, combine='nested', concat_dim='time', decode_cf=True)

# 2. If 'valid_time' is the original time dim, rename it cleanly to 'time'
if 'valid_time' in rain.dims:
    # a) rename the dimension
    rain = rain.rename_dims({'valid_time': 'time'})
    # b) rename the coordinate variable of the same name
    rain = rain.rename_vars({'valid_time': 'time'})
    # c) convert that new 'time' coordinate to datetime64
    rain = rain.assign_coords(time=pd.to_datetime(rain['time'].values))

# 3. Sort chronologically and convert precipitation from m → mm
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

# 4. Read InSAR dates from your HDF5 timeseries
with h5py.File('/content/drive/MyDrive/minty_output/location4/timeseries.h5','r') as f:
    insar_dates = [d.decode() for d in f['date'][:]]
insar_dt = pd.to_datetime(insar_dates, format='%Y%m%d')

# 5. Load your velocity GeoTIFF as the template grid
tpl_path = '/content/velocity_aligned_L4.tif'
with rasterio.open(tpl_path) as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_w, tpl_h  = tpl.width, tpl.height

# 6. Loop: compute 12-day sums, reproject, and export GeoTIFFs
out_dir = '/content/drive/MyDrive/Updated_location4/rainfall_12day'
os.makedirs(out_dir, exist_ok=True)

for dt in insar_dt:
    window = rain.sel(time=slice(dt - pd.Timedelta(days=11), dt))
    if window.time.size < 12:
        print(f"Skipping {dt.date()}: only {window.time.size} days")
        continue

    # sum the daily totals
    arr12 = window['tp'].sum(dim='time').values.astype('float32')

    # prepare an empty array on the template grid
    out_arr = np.full((tpl_h, tpl_w), np.nan, dtype='float32')

    # build source transform from the rainfall coords
    lat = window.latitude.values
    lon = window.longitude.values
    src_transform = from_bounds(
        lon.min(), lat.min(), lon.max(), lat.max(),
        len(lon), len(lat)
    )

    # reproject into the InSAR grid
    reproject(
        source       = arr12,
        destination  = out_arr,
        src_transform= src_transform,
        src_crs      = 'EPSG:4326',
        dst_transform= tpl_transform,
        dst_crs      = tpl_crs,
        resampling   = Resampling.bilinear
    )

    # write out the GeoTIFF
    out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
    with rasterio.open(os.path.join(out_dir, out_name), 'w', **tpl_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Wrote {out_name}")

print(f"Completed exports for {len(insar_dt)} dates.")

ValueError: Cannot rename valid_time to time because time already exists. Try using swap_dims instead.

In [ ]:
import os
import h5py
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# 1. Load and process each ERA5 NetCDF file individually
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]

# Process each file separately to avoid dimension conflicts
datasets = []
for file in era5_files:
    ds = xr.open_dataset(file, decode_cf=True)

    # Fix time dimension if it's named 'valid_time'
    if 'valid_time' in ds.dims and 'valid_time' not in ds.coords:
        # If valid_time is a dimension but not a coordinate, rename it
        ds = ds.rename_dims({'valid_time': 'time'})
    elif 'valid_time' in ds.coords:
        # If valid_time is a coordinate, swap it properly
        if 'time' in ds.coords:
            ds = ds.drop_vars('time')
        ds = ds.swap_dims({'valid_time': 'time'})
        ds = ds.rename({'valid_time': 'time'})

    # Ensure time is a proper datetime coordinate
    if 'time' in ds.coords:
        ds = ds.assign_coords(time=pd.to_datetime(ds.time.values))

    datasets.append(ds)

# 2. Concatenate the processed datasets
rain = xr.concat(datasets, dim='time')

# 3. Sort by time and convert precipitation to millimeters
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

print(f"Final rainfall dataset shape: {rain.tp.shape}")
print(f"Time range: {rain.time.values[0]} to {rain.time.values[-1]}")

# 4. Read InSAR dates
with h5py.File('/content/drive/MyDrive/minty_output/location4/timeseries.h5', 'r') as f:
    insar_dates = [d.decode() for d in f['date'][:]]
insar_dt = pd.to_datetime(insar_dates, format='%Y%m%d')

print(f"Found {len(insar_dt)} InSAR dates")

# 5. Load velocity_aligned_L4.tif as the template grid
tpl_path = '/content/velocity_aligned_L4.tif'
with rasterio.open(tpl_path) as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_w, tpl_h  = tpl.width, tpl.height

print(f"Template grid: {tpl_w} x {tpl_h}")

# 6. Compute and export 12-day sums for each InSAR date
out_dir = '/content/drive/MyDrive/Updated_location4/rainfall_12day'
os.makedirs(out_dir, exist_ok=True)

successful_exports = 0

for i, dt in enumerate(insar_dt):
    try:
        # Define 12-day window ending on InSAR date
        start_date = dt - pd.Timedelta(days=11)
        window = rain.sel(time=slice(start_date, dt))

        if window.time.size < 12:
            print(f"Skipping {dt.date()}: only {window.time.size} days available")
            continue

        # Sum the 12-day daily totals
        arr12 = window['tp'].sum(dim='time').values.astype('float32')

        # Prepare an empty array on the template grid
        out_arr = np.full((tpl_h, tpl_w), np.nan, dtype='float32')

        # Build the source transform from rainfall coords
        lat = window.latitude.values
        lon = window.longitude.values
        src_transform = from_bounds(
            lon.min(), lat.min(), lon.max(), lat.max(),
            len(lon), len(lat)
        )

        # Reproject rainfall sum into InSAR grid
        reproject(
            source        = arr12,
            destination   = out_arr,
            src_transform = src_transform,
            src_crs       = 'EPSG:4326',
            dst_transform = tpl_transform,
            dst_crs       = tpl_crs,
            resampling    = Resampling.bilinear
        )

        # Write the GeoTIFF
        out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
        out_path = os.path.join(out_dir, out_name)

        with rasterio.open(out_path, 'w', **tpl_meta) as dst:
            dst.write(out_arr, 1)

        print(f"Wrote {out_name} - Range: {np.nanmin(out_arr):.2f} to {np.nanmax(out_arr):.2f} mm")
        successful_exports += 1

    except Exception as e:
        print(f"Error processing {dt.date()}: {e}")
        continue

print(f"\nCompleted exports for {successful_exports}/{len(insar_dt)} InSAR dates.")
print(f"Files saved to: {out_dir}")

Final rainfall dataset shape: (550, 9, 13)
Time range: 2023-07-01T00:00:00.000000000 to 2024-12-31T00:00:00.000000000


<ipython-input-14-b884e0454b51>:30: UserWarning: rename 'valid_time' to 'time' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  ds = ds.rename({'valid_time': 'time'})
<ipython-input-14-b884e0454b51>:30: UserWarning: rename 'valid_time' to 'time' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  ds = ds.rename({'valid_time': 'time'})


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/content/drive/MyDrive/minty_output/location4/timeseries.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
import os
import h5py
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# 1. Load and process each ERA5 NetCDF file individually
era5_files = [
    '/content/drive/MyDrive/rainfall_2023.nc',
    '/content/drive/MyDrive/rainfall_2024.nc'
]

# Process each file separately to avoid dimension conflicts
datasets = []
for file in era5_files:
    ds = xr.open_dataset(file, decode_cf=True)

    # Fix time dimension using swap_dims (proper method)
    if 'valid_time' in ds.dims:
        # Drop any conflicting time coordinate first
        if 'time' in ds.coords and ds.coords['time'].shape != (ds.dims['valid_time'],):
            ds = ds.drop_vars('time')

        # Use swap_dims to properly convert valid_time to time dimension
        ds = ds.swap_dims({'valid_time': 'time'})

        # Assign proper datetime coordinate and drop old valid_time
        ds = ds.assign_coords(time=pd.to_datetime(ds.coords['valid_time'].values))
        ds = ds.drop_vars('valid_time')

    datasets.append(ds)

# 2. Concatenate the processed datasets
rain = xr.concat(datasets, dim='time')

# 3. Sort by time and convert precipitation to millimeters
rain = rain.sortby('time')
rain['tp'] = rain['tp'] * 1000

print(f"Final rainfall dataset shape: {rain.tp.shape}")
print(f"Time range: {rain.time.values[0]} to {rain.time.values[-1]}")

# 4. Read InSAR dates
with h5py.File('/content/drive/MyDrive/mintpy_output/location4/timeseries.h5', 'r') as f:
    insar_dates = [d.decode() for d in f['date'][:]]
insar_dt = pd.to_datetime(insar_dates, format='%Y%m%d')

print(f"Found {len(insar_dt)} InSAR dates")

# 5. Load velocity_aligned_L4.tif as the template grid
tpl_path = '/content/velocity_aligned_L4.tif'
with rasterio.open(tpl_path) as tpl:
    tpl_meta      = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs       = tpl.crs
    tpl_w, tpl_h  = tpl.width, tpl.height

print(f"Template grid: {tpl_w} x {tpl_h}")

# 6. Compute and export 12-day sums for each InSAR date
out_dir = '/content/drive/MyDrive/Updated_location4/rainfall_12day'
os.makedirs(out_dir, exist_ok=True)

successful_exports = 0

for i, dt in enumerate(insar_dt):
    try:
        # Define 12-day window ending on InSAR date
        start_date = dt - pd.Timedelta(days=11)
        window = rain.sel(time=slice(start_date, dt))

        if window.time.size < 12:
            print(f"Skipping {dt.date()}: only {window.time.size} days available")
            continue

        # Sum the 12-day daily totals
        arr12 = window['tp'].sum(dim='time').values.astype('float32')

        # Prepare an empty array on the template grid
        out_arr = np.full((tpl_h, tpl_w), np.nan, dtype='float32')

        # Build the source transform from rainfall coords
        lat = window.latitude.values
        lon = window.longitude.values
        src_transform = from_bounds(
            lon.min(), lat.min(), lon.max(), lat.max(),
            len(lon), len(lat)
        )

        # Reproject rainfall sum into InSAR grid
        reproject(
            source        = arr12,
            destination   = out_arr,
            src_transform = src_transform,
            src_crs       = 'EPSG:4326',
            dst_transform = tpl_transform,
            dst_crs       = tpl_crs,
            resampling    = Resampling.bilinear
        )

        # Write the GeoTIFF
        out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
        out_path = os.path.join(out_dir, out_name)

        with rasterio.open(out_path, 'w', **tpl_meta) as dst:
            dst.write(out_arr, 1)

        print(f"Wrote {out_name} - Range: {np.nanmin(out_arr):.2f} to {np.nanmax(out_arr):.2f} mm")
        successful_exports += 1

    except Exception as e:
        print(f"Error processing {dt.date()}: {e}")
        continue

print(f"\nCompleted exports for {successful_exports}/{len(insar_dt)} InSAR dates.")
print(f"Files saved to: {out_dir}")

Final rainfall dataset shape: (550, 9, 13)
Time range: 2023-07-01T00:00:00.000000000 to 2024-12-31T00:00:00.000000000
Found 38 InSAR dates
Template grid: 3549 x 2605
Skipping 2023-07-07: only 7 days available
Wrote rain12_20230719.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20230731.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20230812.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20230824.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20230905.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20230917.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20231011.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20231023.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20231104.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20231116.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20240127.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20240208.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20240220.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20240315.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20240327.tif - Range: 0.00 to 0.00 mm
Wrote rain12_20240408.tif 

In [ ]:
# 1. Check the original rainfall data values and spatial extent
print("=== RAINFALL DATA DIAGNOSTICS ===")
print(f"Original rainfall data shape: {rain.tp.shape}")
print(f"Rainfall coordinates:")
print(f"  Latitude range: {rain.latitude.values.min():.3f} to {rain.latitude.values.max():.3f}")
print(f"  Longitude range: {rain.longitude.values.min():.3f} to {rain.longitude.values.max():.3f}")

# Check actual data values
print(f"\nOriginal tp values (first time step):")
sample_data = rain.tp.isel(time=0).values
print(f"  Min: {sample_data.min():.6f}")
print(f"  Max: {sample_data.max():.6f}")
print(f"  Mean: {sample_data.mean():.6f}")
print(f"  Non-zero count: {np.count_nonzero(sample_data)}")

# 2. Check InSAR grid extent for comparison
with rasterio.open('/content/velocity_aligned_L4.tif') as src:
    bounds = src.bounds
    print(f"\n=== INSAR TEMPLATE GRID ===")
    print(f"InSAR bounds: {bounds}")
    print(f"  West: {bounds.left:.6f}, East: {bounds.right:.6f}")
    print(f"  South: {bounds.bottom:.6f}, North: {bounds.top:.6f}")

# 3. Check coordinate system compatibility
print(f"\n=== COORDINATE OVERLAP CHECK ===")
rain_west, rain_east = rain.longitude.values.min(), rain.longitude.values.max()
rain_south, rain_north = rain.latitude.values.min(), rain.latitude.values.max()
insar_west, insar_east = bounds.left, bounds.right
insar_south, insar_north = bounds.bottom, bounds.top

print(f"Longitude overlap: Rain[{rain_west:.3f}, {rain_east:.3f}] vs InSAR[{insar_west:.3f}, {insar_east:.3f}]")
print(f"Latitude overlap: Rain[{rain_south:.3f}, {rain_north:.3f}] vs InSAR[{insar_south:.3f}, {insar_north:.3f}]")

# Check if there's any spatial overlap
lon_overlap = max(0, min(rain_east, insar_east) - max(rain_west, insar_west))
lat_overlap = max(0, min(rain_north, insar_north) - max(rain_south, insar_south))
print(f"Spatial overlap: {lon_overlap:.6f}° longitude, {lat_overlap:.6f}° latitude")

if lon_overlap == 0 or lat_overlap == 0:
    print("⚠️  WARNING: No spatial overlap between rainfall and InSAR grids!")

# 4. Test a manual 12-day sum to verify the calculation
print(f"\n=== MANUAL CALCULATION TEST ===")
test_date = insar_dt[5]  # Pick a middle date
start = test_date - pd.Timedelta(days=11)
window_test = rain.sel(time=slice(start, test_date))
manual_sum = window_test['tp'].sum(dim='time')
print(f"Test date: {test_date.date()}")
print(f"Manual 12-day sum range: {manual_sum.values.min():.6f} to {manual_sum.values.max():.6f}")

=== RAINFALL DATA DIAGNOSTICS ===
Original rainfall data shape: (550, 9, 13)
Rainfall coordinates:
  Latitude range: 30.870 to 32.870
  Longitude range: 75.660 to 78.660

Original tp values (first time step):
  Min: 0.056003
  Max: 6.680228
  Mean: 1.770350
  Non-zero count: 117

=== INSAR TEMPLATE GRID ===
InSAR bounds: BoundingBox(left=492600.0, bottom=1177800.0, right=658360.0, top=1399000.0)
  West: 492600.000000, East: 658360.000000
  South: 1177800.000000, North: 1399000.000000

=== COORDINATE OVERLAP CHECK ===
Longitude overlap: Rain[75.660, 78.660] vs InSAR[492600.000, 658360.000]
Latitude overlap: Rain[30.870, 32.870] vs InSAR[1177800.000, 1399000.000]
Spatial overlap: 0.000000° longitude, 0.000000° latitude
⚠️  WARNING: No spatial overlap between rainfall and InSAR grids!

=== MANUAL CALCULATION TEST ===
Test date: 2023-09-05
Manual 12-day sum range: 1.254708 to 53.702026


In [ ]:
import rasterio
from rasterio.crs import CRS
import numpy as np

# 1. Fix the velocity file CRS (should be UTM 43N, not WGS84)
print("=== FIXING VELOCITY FILE CRS ===")
with rasterio.open('/content/velocity_aligned_L4.tif') as src:
    # Read the data and metadata
    data = src.read(1)
    profile = src.profile.copy()

# Update the CRS to the correct UTM zone
profile.update(crs=CRS.from_epsg(32643))  # UTM Zone 43N

# Write the corrected velocity file
with rasterio.open('/content/velocity_aligned_L4_corrected.tif', 'w', **profile) as dst:
    dst.write(data, 1)

print("Created velocity_aligned_L4_corrected.tif with correct CRS (EPSG:32643)")

# 2. Verify the fix
with rasterio.open('/content/velocity_aligned_L4_corrected.tif') as src:
    print(f"Corrected CRS: {src.crs}")
    print(f"Bounds: {src.bounds}")

# 3. Now reproject rainfall to the corrected UTM grid
from rasterio.warp import reproject, Resampling, transform_bounds

# Get the corrected template
with rasterio.open('/content/velocity_aligned_L4_corrected.tif') as tpl:
    tpl_meta = tpl.meta.copy()
    tpl_transform = tpl.transform
    tpl_crs = tpl.crs
    tpl_w, tpl_h = tpl.width, tpl.height

print(f"\n=== CORRECTED TEMPLATE GRID ===")
print(f"CRS: {tpl_crs}")
print(f"Transform: {tpl_transform}")
print(f"Size: {tpl_w} x {tpl_h}")

# 4. Transform rainfall bounds to UTM for verification
rain_bounds_utm = transform_bounds(
    CRS.from_epsg(4326),
    CRS.from_epsg(32643),
    rain.longitude.values.min(), rain.latitude.values.min(),
    rain.longitude.values.max(), rain.latitude.values.max()
)
print(f"Rainfall bounds in UTM: {rain_bounds_utm}")
print(f"InSAR bounds in UTM: {tpl.bounds}")

# Check overlap in UTM coordinates
utm_lon_overlap = max(0, min(rain_bounds_utm[2], tpl.bounds.right) - max(rain_bounds_utm[0], tpl.bounds.left))
utm_lat_overlap = max(0, min(rain_bounds_utm[3], tpl.bounds.top) - max(rain_bounds_utm[1], tpl.bounds.bottom))
print(f"UTM overlap: {utm_lon_overlap:.1f}m longitude, {utm_lat_overlap:.1f}m latitude")

if utm_lon_overlap > 0 and utm_lat_overlap > 0:
    print("✅ Good! There is spatial overlap in UTM coordinates.")
else:
    print("❌ Still no overlap - need to check coordinate systems further.")

=== FIXING VELOCITY FILE CRS ===
Created velocity_aligned_L4_corrected.tif with correct CRS (EPSG:32643)
Corrected CRS: EPSG:32643
Bounds: BoundingBox(left=492600.0, bottom=1177800.0, right=658360.0, top=1399000.0)

=== CORRECTED TEMPLATE GRID ===
CRS: EPSG:32643
Transform: | 80.00, 0.00, 492600.00|
| 0.00,-80.00, 1399000.00|
| 0.00, 0.00, 1.00|
Size: 2072 x 2765
Rainfall bounds in UTM: (561745.27344063, 3415381.272809315, 849992.3804161396, 3642817.3132035905)
InSAR bounds in UTM: BoundingBox(left=492600.0, bottom=1177800.0, right=658360.0, top=1399000.0)
UTM overlap: 96614.7m longitude, 0.0m latitude
❌ Still no overlap - need to check coordinate systems further.


In [ ]:
# 1. Check what latitude the InSAR northing values actually correspond to
from rasterio.warp import transform
from pyproj import CRS, Transformer

# Transform InSAR bounds back to geographic coordinates to see where they really point
transformer = Transformer.from_crs(CRS.from_epsg(32643), CRS.from_epsg(4326), always_xy=True)

# Convert InSAR corners to lat/lon
insar_bounds = tpl.bounds
sw_lon, sw_lat = transformer.transform(insar_bounds.left, insar_bounds.bottom)
ne_lon, ne_lat = transformer.transform(insar_bounds.right, insar_bounds.top)

print(f"=== INSAR ACTUAL GEOGRAPHIC LOCATION ===")
print(f"InSAR SW corner: {sw_lon:.6f}°E, {sw_lat:.6f}°N")
print(f"InSAR NE corner: {ne_lon:.6f}°E, {ne_lat:.6f}°N")
print(f"Expected location: 75.66-78.66°E, 30.87-32.87°N")

# 2. Check if InSAR should be in a different UTM zone
print(f"\n=== UTM ZONE CHECK ===")
center_lon = (75.66 + 78.66) / 2
center_lat = (30.87 + 32.87) / 2
print(f"Study area center: {center_lon:.3f}°E, {center_lat:.3f}°N")

# Calculate correct UTM zone
utm_zone = int((center_lon + 180) / 6) + 1
print(f"Correct UTM zone should be: {utm_zone}N (EPSG:326{utm_zone:02d})")

# 3. Try transforming rainfall to the InSAR's apparent location for comparison
rain_center_utm = transform_bounds(
    CRS.from_epsg(4326), CRS.from_epsg(32643),
    center_lon, center_lat, center_lon, center_lat
)
print(f"Expected rainfall center in UTM 43N: {rain_center_utm}")

=== INSAR ACTUAL GEOGRAPHIC LOCATION ===
InSAR SW corner: 74.932339°E, 10.654694°N
InSAR NE corner: 76.458175°E, 12.651153°N
Expected location: 75.66-78.66°E, 30.87-32.87°N

=== UTM ZONE CHECK ===
Study area center: 77.160°E, 31.870°N
Correct UTM zone should be: 43N (EPSG:32643)
Expected rainfall center in UTM 43N: (704332.4629888427, 3528060.5243963283, 704332.4629888427, 3528060.5243963283)


In [ ]:
from affine import Affine
from pyproj import CRS, Transformer

# 1. Calculate the geographic offset needed
current_center_lon = (74.932339 + 76.458175) / 2  # ~75.695°E
current_center_lat = (10.654694 + 12.651153) / 2  # ~11.653°N
expected_center_lon = 77.160  # From your rainfall data
expected_center_lat = 31.870  # From your rainfall data

lon_offset = expected_center_lon - current_center_lon  # ~+1.465°
lat_offset = expected_center_lat - current_center_lat  # ~+20.217°

print(f"=== GEOLOCATION CORRECTION NEEDED ===")
print(f"Longitude offset: {lon_offset:+.3f}°")
print(f"Latitude offset: {lat_offset:+.3f}°")

# 2. Convert the offset to UTM meters
transformer = Transformer.from_crs(CRS.from_epsg(4326), CRS.from_epsg(32643), always_xy=True)

# Current center in UTM
current_utm_x, current_utm_y = transformer.transform(current_center_lon, current_center_lat)
# Expected center in UTM
expected_utm_x, expected_utm_y = transformer.transform(expected_center_lon, expected_center_lat)

utm_x_offset = expected_utm_x - current_utm_x
utm_y_offset = expected_utm_y - current_utm_y

print(f"UTM offset needed:")
print(f"  Easting: {utm_x_offset:+.0f} meters")
print(f"  Northing: {utm_y_offset:+.0f} meters")

# 3. Apply the correction to the transform
with rasterio.open('/content/velocity_aligned_L4_corrected.tif') as src:
    old_transform = src.transform
    data = src.read(1)
    profile = src.profile.copy()

# Create corrected transform by shifting the origin
corrected_transform = Affine(
    old_transform.a,  # same pixel size in X
    old_transform.b,  # same rotation
    old_transform.c + utm_x_offset,  # shift X origin
    old_transform.d,  # same rotation
    old_transform.e,  # same pixel size in Y
    old_transform.f + utm_y_offset   # shift Y origin
)

print(f"\nOriginal transform: {old_transform}")
print(f"Corrected transform: {corrected_transform}")

# 4. Write the geocorrected velocity file
profile.update(transform=corrected_transform)
with rasterio.open('/content/velocity_aligned_L4_geocorrected.tif', 'w', **profile) as dst:
    dst.write(data, 1)

print("✅ Created velocity_aligned_L4_geocorrected.tif with correct geolocation")

# 5. Verify the correction worked
with rasterio.open('/content/velocity_aligned_L4_geocorrected.tif') as src:
    bounds = src.bounds

# Transform back to geographic to verify
sw_lon, sw_lat = transformer.transform(bounds.left, bounds.bottom, direction='INVERSE')
ne_lon, ne_lat = transformer.transform(bounds.right, bounds.top, direction='INVERSE')

print(f"\n=== VERIFICATION ===")
print(f"Corrected InSAR location:")
print(f"  SW corner: {sw_lon:.6f}°E, {sw_lat:.6f}°N")
print(f"  NE corner: {ne_lon:.6f}°E, {ne_lat:.6f}°N")
print(f"Expected rainfall area: 75.66-78.66°E, 30.87-32.87°N")

# Check overlap
lon_overlap = max(0, min(ne_lon, 78.66) - max(sw_lon, 75.66))
lat_overlap = max(0, min(ne_lat, 32.87) - max(sw_lat, 30.87))
print(f"Geographic overlap: {lon_overlap:.3f}° longitude, {lat_overlap:.3f}° latitude")

if lon_overlap > 0 and lat_overlap > 0:
    print("🎉 SUCCESS! InSAR and rainfall now have spatial overlap!")
else:
    print("❌ Still no overlap - need further adjustment")

=== GEOLOCATION CORRECTION NEEDED ===
Longitude offset: +1.465°
Latitude offset: +20.217°
UTM offset needed:
  Easting: +128550 meters
  Northing: +2239793 meters

Original transform: | 80.00, 0.00, 492600.00|
| 0.00,-80.00, 1399000.00|
| 0.00, 0.00, 1.00|
Corrected transform: | 80.00, 0.00, 621150.25|
| 0.00,-80.00, 3638792.58|
| 0.00, 0.00, 1.00|
✅ Created velocity_aligned_L4_geocorrected.tif with correct geolocation

=== VERIFICATION ===
Corrected InSAR location:
  SW corner: 76.267478°E, 30.885429°N
  NE corner: 78.065530°E, 32.849717°N
Expected rainfall area: 75.66-78.66°E, 30.87-32.87°N
Geographic overlap: 1.798° longitude, 1.964° latitude
🎉 SUCCESS! InSAR and rainfall now have spatial overlap!


In [ ]:
import os
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling

# Use the geocorrected velocity as the master template
template_path = '/content/velocity_aligned_L4_geocorrected.tif'

with rasterio.open(template_path) as tpl:
    template_meta = tpl.meta.copy()
    template_transform = tpl.transform
    template_crs = tpl.crs
    template_width = tpl.width
    template_height = tpl.height

print(f"=== REGENERATING ALIGNED PRODUCTS ===")
print(f"Using template: {template_path}")
print(f"CRS: {template_crs}")
print(f"Size: {template_width} x {template_height}")

# 1. Regenerate trend map using the geocorrected template
print("\n1. Regenerating trend map...")
with h5py.File('/content/drive/MyDrive/mintpy_output/location4/timeseries.h5', 'r') as f:
    ts = f['timeseries'][:]
    dates = [np.datetime64(d.decode('utf-8'), 'D') for d in f['date'][:]]

# Recalculate trend with correct geolocation
t = np.arange(len(dates), dtype=float)
mean_t = t.mean()
var_t = ((t - mean_t) ** 2).sum()
trend = ((ts * (t[:, None, None] - mean_t)).sum(axis=0)) / var_t

# Apply the same geolocation correction to trend
with rasterio.open('/content/trend_L4_geocorrected.tif', 'w', **template_meta) as dst:
    dst.write(trend.astype('float32'), 1)

print("✅ Created trend_L4_geocorrected.tif")

# 2. Regenerate DEM using the geocorrected template
print("\n2. Regenerating aligned DEM...")
dem_src_path = '/content/S1AA_20220101T004848_20220113T004847_VVP012_INT80_G_weF_5C84_dem.tif'

with rasterio.open(dem_src_path) as src:
    aligned_dem = np.empty((template_height, template_width), dtype=np.float32)
    aligned_dem.fill(np.nan)

    reproject(
        source=src.read(1),
        destination=aligned_dem,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=template_transform,
        dst_crs=template_crs,
        resampling=Resampling.bilinear
    )

with rasterio.open('/content/DEM_L4_geocorrected.tif', 'w', **template_meta) as dst:
    dst.write(aligned_dem, 1)

print("✅ Created DEM_L4_geocorrected.tif")

# 3. Regenerate slope and aspect from the geocorrected DEM
print("\n3. Regenerating slope and aspect...")
dx = template_transform.a
dy = -template_transform.e

dzdx, dzdy = np.gradient(aligned_dem, dx, dy)
slope = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))
aspect = (np.degrees(np.arctan2(dzdy, -dzdx)) + 180) % 360

for arr, name in [(slope, '/content/slope_L4_geocorrected.tif'),
                  (aspect, '/content/aspect_L4_geocorrected.tif')]:
    with rasterio.open(name, 'w', **template_meta) as dst:
        dst.write(arr.astype('float32'), 1)

print("✅ Created slope_L4_geocorrected.tif and aspect_L4_geocorrected.tif")

=== REGENERATING ALIGNED PRODUCTS ===
Using template: /content/velocity_aligned_L4_geocorrected.tif
CRS: EPSG:32643
Size: 2072 x 2765

1. Regenerating trend map...
✅ Created trend_L4_geocorrected.tif

2. Regenerating aligned DEM...
✅ Created DEM_L4_geocorrected.tif

3. Regenerating slope and aspect...
✅ Created slope_L4_geocorrected.tif and aspect_L4_geocorrected.tif


In [ ]:
# Re-export all 12-day rainfall sums using the geocorrected template
print("\n=== RE-EXPORTING RAINFALL GEOTIFFS ===")

output_dir = '/content/drive/MyDrive/Updated_location4/rainfall_12day_final'
os.makedirs(output_dir, exist_ok=True)

successful_exports = 0

for i, dt in enumerate(insar_dt):
    try:
        # Define 12-day window ending on InSAR date
        start_date = dt - pd.Timedelta(days=11)
        window = rain.sel(time=slice(start_date, dt))

        if window.time.size < 12:
            print(f"Skipping {dt.date()}: only {window.time.size} days available")
            continue

        # Sum the 12-day daily totals (already in mm)
        arr12 = window['tp'].sum(dim='time').values.astype('float32')

        # Prepare output array on the geocorrected template grid
        out_arr = np.full((template_height, template_width), np.nan, dtype='float32')

        # Build source transform from rainfall coords (WGS84)
        lat = window.latitude.values
        lon = window.longitude.values
        src_transform = from_bounds(
            lon.min(), lat.min(), lon.max(), lat.max(),
            len(lon), len(lat)
        )

        # Reproject from WGS84 to geocorrected UTM grid
        reproject(
            source        = arr12,
            destination   = out_arr,
            src_transform = src_transform,
            src_crs       = CRS.from_epsg(4326),
            dst_transform = template_transform,
            dst_crs       = template_crs,
            resampling    = Resampling.bilinear
        )

        # Write the GeoTIFF
        out_name = f'rain12_{dt.strftime("%Y%m%d")}.tif'
        out_path = os.path.join(output_dir, out_name)

        with rasterio.open(out_path, 'w', **template_meta) as dst:
            dst.write(out_arr, 1)

        # Check if we got real rainfall values
        valid_data = out_arr[~np.isnan(out_arr)]
        if len(valid_data) > 0:
            print(f"✅ {out_name} - Range: {valid_data.min():.2f} to {valid_data.max():.2f} mm")
            successful_exports += 1
        else:
            print(f"❌ {out_name} - No valid data")

    except Exception as e:
        print(f"Error processing {dt.date()}: {e}")
        continue

print(f"\n🎉 Successfully exported {successful_exports}/{len(insar_dt)} rainfall GeoTIFFs!")
print(f"Files saved to: {output_dir}")


=== RE-EXPORTING RAINFALL GEOTIFFS ===
Skipping 2023-07-07: only 7 days available
✅ rain12_20230719.tif - Range: 0.00 to 411.26 mm
✅ rain12_20230731.tif - Range: 0.00 to 221.85 mm
✅ rain12_20230812.tif - Range: 0.00 to 238.13 mm
✅ rain12_20230824.tif - Range: 0.00 to 359.59 mm
✅ rain12_20230905.tif - Range: 0.00 to 53.70 mm
✅ rain12_20230917.tif - Range: 0.00 to 136.63 mm
✅ rain12_20231011.tif - Range: 0.00 to 38.96 mm
✅ rain12_20231023.tif - Range: 0.00 to 39.63 mm
✅ rain12_20231104.tif - Range: 0.00 to 11.15 mm
✅ rain12_20231116.tif - Range: 0.00 to 18.82 mm
✅ rain12_20240127.tif - Range: 0.00 to 27.47 mm
✅ rain12_20240208.tif - Range: 0.00 to 159.34 mm
✅ rain12_20240220.tif - Range: 0.00 to 127.37 mm
✅ rain12_20240315.tif - Range: 0.00 to 65.09 mm
✅ rain12_20240327.tif - Range: 0.00 to 15.56 mm
✅ rain12_20240408.tif - Range: 0.00 to 77.11 mm
✅ rain12_20240420.tif - Range: 0.00 to 78.30 mm
✅ rain12_20240502.tif - Range: 0.00 to 93.76 mm
✅ rain12_20240514.tif - Range: 0.00 to 49.90 m

In [ ]:
# Verify all geocorrected products have identical metadata
print("\n=== VERIFICATION OF ALL ALIGNED PRODUCTS ===")

products = [
    '/content/velocity_aligned_L4_geocorrected.tif',
    '/content/trend_L4_geocorrected.tif',
    '/content/DEM_L4_geocorrected.tif',
    '/content/slope_L4_geocorrected.tif',
    '/content/aspect_L4_geocorrected.tif'
]

for product in products:
    if os.path.exists(product):
        with rasterio.open(product) as src:
            print(f"{os.path.basename(product)}:")
            print(f"  CRS: {src.crs}")
            print(f"  Size: {src.width}x{src.height}")
            print(f"  Transform: {src.transform}")
            print(f"  Bounds: {src.bounds}")
            print()


=== VERIFICATION OF ALL ALIGNED PRODUCTS ===
velocity_aligned_L4_geocorrected.tif:
  CRS: EPSG:32643
  Size: 2072x2765
  Transform: | 80.00, 0.00, 621150.25|
| 0.00,-80.00, 3638792.58|
| 0.00, 0.00, 1.00|
  Bounds: BoundingBox(left=621150.2536976888, bottom=3417592.5844404334, right=786910.2536976888, top=3638792.5844404334)

trend_L4_geocorrected.tif:
  CRS: EPSG:32643
  Size: 2072x2765
  Transform: | 80.00, 0.00, 621150.25|
| 0.00,-80.00, 3638792.58|
| 0.00, 0.00, 1.00|
  Bounds: BoundingBox(left=621150.2536976888, bottom=3417592.5844404334, right=786910.2536976888, top=3638792.5844404334)

DEM_L4_geocorrected.tif:
  CRS: EPSG:32643
  Size: 2072x2765
  Transform: | 80.00, 0.00, 621150.25|
| 0.00,-80.00, 3638792.58|
| 0.00, 0.00, 1.00|
  Bounds: BoundingBox(left=621150.2536976888, bottom=3417592.5844404334, right=786910.2536976888, top=3638792.5844404334)

slope_L4_geocorrected.tif:
  CRS: EPSG:32643
  Size: 2072x2765
  Transform: | 80.00, 0.00, 621150.25|
| 0.00,-80.00, 3638792.58|


In [ ]:
# Check if your landslide label file is aligned with the other products
print("=== CHECKING LABEL FILE ALIGNMENT ===")
label_path = '/content/drive/MyDrive/Updated_location4/ls_labels_L4.tif'

with rasterio.open(label_path) as src:
    print(f"ls_label_L4.tiff:")
    print(f"  CRS: {src.crs}")
    print(f"  Size: {src.width}x{src.height}")
    print(f"  Transform: {src.transform}")
    print(f"  Bounds: {src.bounds}")

    # Check label values
    labels = src.read(1)
    unique_vals = np.unique(labels[~np.isnan(labels)])
    print(f"  Unique label values: {unique_vals}")
    print(f"  Landslide pixels: {np.sum(labels == 1)}")
    print(f"  Non-landslide pixels: {np.sum(labels == 0)}")

# Check if it matches the template
template_path = '/content/velocity_aligned_L4_geocorrected.tif'
with rasterio.open(template_path) as tpl:
    if (src.crs == tpl.crs and src.width == tpl.width and
        src.height == tpl.height and src.transform == tpl.transform):
        print("✅ Label file is perfectly aligned!")
    else:
        print("❌ Label file needs alignment")

=== CHECKING LABEL FILE ALIGNMENT ===
ls_label_L4.tiff:
  CRS: EPSG:32643
  Size: 250x269
  Transform: | 9.30, 0.00, 623544.23|
| 0.00,-9.30, 1269828.19|
| 0.00, 0.00, 1.00|
  Bounds: BoundingBox(left=623544.2348152425, bottom=1267326.4913756805, right=625869.2348152425, top=1269828.1913756805)
  Unique label values: [0 1]
  Landslide pixels: 3367
  Non-landslide pixels: 63883
❌ Label file needs alignment
